# NLT Replication: Visual Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sage-is/NLT-Replication-Study/blob/develop/notebooks/analysis_visuals.ipynb)

Purpose: visualize accuracy, variance, errors, aborted runs, token usage, and NLT gains.
- Loads aggregated_results.csv with token usage columns
- Summarizes by approach, model, scenario, and perturbation
- Generates charts for the replication study
- Saves PNGs to results/figures/

**Runs anywhere**: local (uv/pip venv), Google Colab, and Kaggle notebooks.
- Locally: run from the repo root or `notebooks/` — `aggregated_results.csv` is found on disk.
- Colab/Kaggle: the repo isn't checked out, so the data-loading cell below falls back to
  downloading `aggregated_results.csv` straight from GitHub. On Kaggle, make sure the kernel's
  **Internet** setting is turned on (Settings → Internet → On) or the download will fail.

Run order: install deps → load data → summaries → plots

In [ ]:
# Optional dependency install (uses pip via sys.executable — works in local venvs, Colab, and Kaggle alike)
import importlib.util
import subprocess
import sys

def ensure(pkg: str):
    if importlib.util.find_spec(pkg) is None:
        print(f'Installing {pkg} ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
    else:
        print(f'{pkg} already available')

for pkg in ['pandas', 'matplotlib', 'seaborn']:
    ensure(pkg)

In [ ]:
import os
import urllib.request
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def detect_environment():
    if 'google.colab' in sys.modules:
        return 'colab'
    if os.environ.get('KAGGLE_KERNEL_RUN_TYPE') or Path('/kaggle/input').exists():
        return 'kaggle'
    return 'local'

ENV = detect_environment()
print(f'Detected environment: {ENV}')

RAW_CSV_URL = 'https://raw.githubusercontent.com/Sage-is/NLT-Replication-Study/develop/aggregated_results.csv'

ROOT = Path.cwd()
csv_candidates = [ROOT / 'aggregated_results.csv', ROOT.parent / 'aggregated_results.csv']
CSV_PATH = next((c for c in csv_candidates if c.exists()), None)

if CSV_PATH is None:
    # Colab/Kaggle: the repo isn't checked out locally, so fetch the CSV directly from GitHub
    CSV_PATH = ROOT / 'aggregated_results.csv'
    print(f'Local CSV not found, downloading from {RAW_CSV_URL} ...')
    urllib.request.urlretrieve(RAW_CSV_URL, CSV_PATH)

# Figures always land next to aggregated_results.csv (repo root), regardless of
# whether this notebook is run from the repo root, notebooks/, or a Colab/Kaggle cwd.
PROJECT_ROOT = CSV_PATH.parent
FIG_DIR = PROJECT_ROOT / 'results' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

print('Using CSV:', CSV_PATH)
print('Saving figures to:', FIG_DIR)
sns.set_theme(style='whitegrid', palette='colorblind')

In [ ]:
# Load data
df = pd.read_csv(CSV_PATH)

# Convert numeric columns
num_cols = ['accuracy', 'variance', 'total', 'errors', 'valid_trials', 'total_tokens', 'prompt_tokens', 'completion_tokens']
for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

df['error_rate'] = df['errors'] / df['total'].where(df['total'] > 0, 1)
df['success'] = (df['accuracy'] > 0) & (df['error_rate'] <= 0.75)

print(f"Loaded {len(df)} rows")
print(f"Columns: {df.columns.tolist()}")
df.head()

## Summary Statistics

In [ ]:
# Summary by approach
by_approach = df.groupby('approach').agg({
    'accuracy': 'mean',
    'variance': 'mean',
    'errors': 'sum',
    'total_tokens': 'sum',
    'prompt_tokens': 'sum',
    'completion_tokens': 'sum'
}).round(4)

print("\n=== BY APPROACH ===")
print(by_approach)

In [ ]:
# Summary by model and approach
by_model = df.groupby(['model_id', 'approach']).agg({
    'accuracy': 'mean',
    'errors': 'sum'
}).round(4)

print("\n=== BY MODEL & APPROACH ===")
print(by_model)

## Visualizations

In [ ]:
# Figure 1: Accuracy by Approach
fig, ax = plt.subplots(figsize=(8, 5))
approach_acc = df.groupby('approach')['accuracy'].mean()
approach_acc.plot(kind='bar', ax=ax, color=['#2ecc71', '#e74c3c'])
ax.set_title('Mean Accuracy by Approach', fontsize=14, fontweight='bold')
ax.set_xlabel('Approach')
ax.set_ylabel('Accuracy')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig1_accuracy_by_approach.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 2: Accuracy by Model (NLT vs Structured)
model_pivot = df.pivot_table(index='model_id', columns='approach', values='accuracy', aggfunc='mean')
model_pivot = model_pivot.sort_values('nlt', ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
model_pivot.plot(kind='bar', ax=ax, color=['#2ecc71', '#e74c3c'])
ax.set_title('Accuracy by Model: NLT vs Structured', fontsize=14, fontweight='bold')
ax.set_xlabel('Model')
ax.set_ylabel('Accuracy')
ax.set_xticklabels([m.split('/')[-1][:20] for m in model_pivot.index], rotation=45, ha='right')
ax.legend(title='Approach')
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig2_accuracy_by_model.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 3: Error Counts by Approach
fig, ax = plt.subplots(figsize=(8, 5))
error_counts = df.groupby('approach')['errors'].sum()
error_counts.plot(kind='bar', ax=ax, color=['#2ecc71', '#e74c3c'])
ax.set_title('Total Errors by Approach', fontsize=14, fontweight='bold')
ax.set_xlabel('Approach')
ax.set_ylabel('Total Errors')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig3_errors_by_approach.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 4: Variance Comparison
fig, ax = plt.subplots(figsize=(8, 5))
variance_data = df.groupby('approach')['variance'].mean()
variance_data.plot(kind='bar', ax=ax, color=['#2ecc71', '#e74c3c'])
ax.set_title('Mean Variance by Approach', fontsize=14, fontweight='bold')
ax.set_xlabel('Approach')
ax.set_ylabel('Variance')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig4_variance_by_approach.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 5: Accuracy by Scenario
scenario_pivot = df.pivot_table(index='scenario', columns='approach', values='accuracy', aggfunc='mean')

fig, ax = plt.subplots(figsize=(8, 5))
scenario_pivot.plot(kind='bar', ax=ax, color=['#2ecc71', '#e74c3c'])
ax.set_title('Accuracy by Scenario: NLT vs Structured', fontsize=14, fontweight='bold')
ax.set_xlabel('Scenario')
ax.set_ylabel('Accuracy')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(title='Approach')
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig5_accuracy_by_scenario.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 6: Perturbation Robustness
pert_pivot = df.pivot_table(index='perturbed', columns='approach', values='accuracy', aggfunc='mean')
pert_pivot.index = ['Non-perturbed', 'Perturbed']

fig, ax = plt.subplots(figsize=(8, 5))
pert_pivot.plot(kind='bar', ax=ax, color=['#2ecc71', '#e74c3c'])
ax.set_title('Perturbation Robustness: NLT vs Structured', fontsize=14, fontweight='bold')
ax.set_xlabel('Prompt Type')
ax.set_ylabel('Accuracy')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(title='Approach')
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig6_perturbation_robustness.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 7: Token Usage Comparison
token_data = df.groupby('approach')[['total_tokens', 'prompt_tokens', 'completion_tokens']].sum()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Total tokens
token_data['total_tokens'].plot(kind='bar', ax=ax1, color=['#2ecc71', '#e74c3c'])
ax1.set_title('Total Token Usage by Approach', fontsize=14, fontweight='bold')
ax1.set_xlabel('Approach')
ax1.set_ylabel('Total Tokens')
ax1.set_xticklabels(ax1.get_xticklabels(), rotation=0)
ax1.grid(axis='y', alpha=0.3)

# Token breakdown
token_data[['prompt_tokens', 'completion_tokens']].plot(kind='bar', ax=ax2, stacked=True)
ax2.set_title('Token Breakdown by Approach', fontsize=14, fontweight='bold')
ax2.set_xlabel('Approach')
ax2.set_ylabel('Tokens')
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=0)
ax2.legend(title='Token Type', labels=['Input', 'Output'])
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig7_token_usage.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 8: NLT Gains by Model
gains_df = df.pivot_table(index='model_id', columns='approach', values='accuracy', aggfunc='mean')
gains_df = gains_df.dropna(subset=['nlt', 'structured'])
gains_df['gain'] = gains_df['nlt'] - gains_df['structured']
gains_df = gains_df.sort_values('gain', ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#e74c3c' if x < 0 else '#2ecc71' for x in gains_df['gain']]
gains_df['gain'].plot(kind='barh', ax=ax, color=colors)
ax.set_title('NLT Accuracy Gain over Structured (by Model)', fontsize=14, fontweight='bold')
ax.set_xlabel('Accuracy Gain (NLT - Structured)')
ax.set_ylabel('Model')
ax.set_yticklabels([m.split('/')[-1][:30] for m in gains_df.index])
ax.axvline(0, color='black', linestyle='--', linewidth=1)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig8_nlt_gains_by_model.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 9: Accuracy Distribution by Approach (box + strip)
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df, x='approach', y='accuracy', hue='approach', palette=['#2ecc71', '#e74c3c'], legend=False, ax=ax)
sns.stripplot(data=df, x='approach', y='accuracy', color='black', alpha=0.35, size=3, jitter=True, ax=ax)
ax.set_title('Accuracy Distribution by Approach', fontsize=14, fontweight='bold')
ax.set_xlabel('Approach')
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig9_accuracy_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 10: Variance Distribution by Approach (box + strip)
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df, x='approach', y='variance', hue='approach', palette=['#2ecc71', '#e74c3c'], legend=False, ax=ax)
sns.stripplot(data=df, x='approach', y='variance', color='black', alpha=0.35, size=3, jitter=True, ax=ax)
ax.set_title('Variance Distribution by Approach', fontsize=14, fontweight='bold')
ax.set_xlabel('Approach')
ax.set_ylabel('Variance')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig10_variance_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 11: Error Counts by Model and Approach
error_by_model = df.groupby(['model_id', 'approach'])['errors'].sum().unstack('approach').fillna(0)
error_by_model = error_by_model.loc[error_by_model.sum(axis=1).sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(12, 6))
error_by_model.plot(kind='bar', ax=ax, color=['#2ecc71', '#e74c3c'])
ax.set_title('Error Counts by Model and Approach', fontsize=14, fontweight='bold')
ax.set_xlabel('Model')
ax.set_ylabel('Total Errors')
ax.set_xticklabels([m.split('/')[-1][:20] for m in error_by_model.index], rotation=45, ha='right')
ax.legend(title='Approach')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig11_errors_by_model.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 12: Aborted Runs by Model
df['aborted_flag'] = df['aborted'].astype(str).str.lower().eq('yes')
aborted_by_model = df.groupby('model_id')['aborted_flag'].sum()
aborted_by_model = aborted_by_model[aborted_by_model > 0].sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
if len(aborted_by_model) > 0:
    aborted_by_model.plot(kind='bar', ax=ax, color='#e67e22')
    ax.set_xticklabels([m.split('/')[-1][:20] for m in aborted_by_model.index], rotation=45, ha='right')
else:
    ax.text(0.5, 0.5, 'No aborted runs recorded', ha='center', va='center', transform=ax.transAxes, fontsize=12)
ax.set_title('Aborted Runs by Model', fontsize=14, fontweight='bold')
ax.set_xlabel('Model')
ax.set_ylabel('Aborted Run Count')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig12_aborted_by_model.png', dpi=300, bbox_inches='tight')
plt.show()

## Complete

12 figures saved to `results/figures/`:
1. Accuracy by Approach
2. Accuracy by Model (NLT vs Structured)
3. Errors by Approach
4. Variance by Approach
5. Accuracy by Scenario
6. Perturbation Robustness
7. Token Usage
8. NLT Gains by Model
9. Accuracy Distribution (box/strip)
10. Variance Distribution (box/strip)
11. Errors by Model and Approach
12. Aborted Runs by Model